# Quickstart

Model, dataset, one batch. Needs the data from `scripts/download_data.py`.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

import yaml, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from src.model import Detector, score
from src.data import ImageDataset, load_rows

cfg = yaml.safe_load(open('configs/train.yaml'))

## Как устроена модель

![](../assets/architecture.svg)

**Идея.** Обычный классификатор картинок. Один проход сети отвечает сразу на два вопроса: сгенерирована ли картинка и каким семейством генераторов.

**1. Сеть.** DINOv3 ConvNeXt-Tiny, 28M параметров. Картинка 512×512 превращается в вектор из 768 чисел: среднее по последней карте признаков.

**2. Классы.** Линейный слой и softmax по классам «реальная, семейство 1, семейство 2, …». Семейство — генераторы одного производителя или одной архитектуры: OpenAI, Google, Flux, SDXL и так далее (`model.families` в конфиге). Новая версия известного семейства попадает в его класс.

$$p(\text{сгенерирована}) = 1 - p(\text{реальная})$$

**3. Лосс.** Кросс-энтропия, она раскладывается на «сгенерирована или нет» и «какое семейство»:

$$-\log p(\text{семейство } j) = -\log p(\text{сгенерирована}) - \log p(\text{семейство } j \mid \text{сгенерирована})$$

**4. InfoNCE** (`train.contrast`, по умолчанию выключен). К кросс-энтропии добавляется контрастивный лосс на векторах картинок: картинки одного класса стягиваются, разных расталкиваются. Вектор $z_i$ сравнивается с векторами своего батча и последних шагов обучения, всего до 4096 штук. $P_i$ — те из них, что одного класса с $z_i$:

$$\mathcal{L}_i = -\frac{1}{|P_i|}\sum_{p \in P_i} \log \frac{\exp(z_i \cdot z_p / \tau)}{\sum_{k \neq i} \exp(z_i \cdot z_k / \tau)}, \qquad \tau = 0.1$$

**5. Выравнивание генераторов** (`train.balance`, по умолчанию выключено). Генераторы очень неравны: у одних 10 000 картинок, у других 200. Картинка генератора с $n$ картинками берётся в эпоху с весом $n^{-\text{balance}}$, поэтому редкие генераторы модель видит чаще.

**6. Калибровка.** После обучения смещение класса «реальная» сдвигается по отложенным реальным картинкам: выше 0.5 остаётся около 1 % из них (`train.alpha`). Поэтому порога рядом с моделью нет: картинка сгенерирована, если $p > 0.5$.

**7. Сравнение вариантов.** Каждый запуск пишет в свою папку `runs/<имя>`. `run_test.py` сводит все проверенные запуски в одну таблицу `runs/compare.md`.

In [ ]:
# model: the backbone and a linear layer over [real, families]
families = cfg['model']['families']
family_of = {g: name for name, generators in families.items() for g in generators}
model = Detector(cfg['model']['backbone'], list(families))
million = lambda part: sum(p.numel() for p in part.parameters()) / 1e6
print(f"backbone {million(model.net):.1f}M parameters, {len(model.classes)} families")

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

In [ ]:
# dataset: whole images resized to 512 x 512, with augmentations; label 0 = real, j = the j-th family
number = {name: j for j, name in enumerate(model.classes, 1)}
rows = [r for r in load_rows('data/openfake_train.csv') if not r['generator'] or r['generator'] in family_of]
for r in rows:
    r['label'] = number[family_of[r['generator']]] if r['generator'] else 0
dataset = ImageDataset(rows, cfg['train']['size'], train=True)
image, label = next(iter(DataLoader(dataset, batch_size=4, shuffle=True)))
image.shape, label

In [ ]:
# forward and loss
logits = model(image.to(device))
loss = F.cross_entropy(logits, label.to(device))
prob, index = score(logits)
logits.shape, prob, [model.classes[i] for i in index], loss.item()

In [ ]:
# the views the model sees
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, im, k in zip(ax, image, label):
    a.imshow(im.permute(1, 2, 0))
    a.set_title('real' if k == 0 else model.classes[k - 1]); a.axis('off')

## Данные

**Обучение: [OpenFake](https://arxiv.org/abs/2509.09495).** Лицензия CC BY-NC 4.0, только для исследований. Реальные фото взяты из LAION и Pexels: публичные люди, новости, мемы с текстом. Для каждого фото нейросеть написала подпись, и по этим подписям картинки сгенерировали 72 модели: GPT Image, Nano Banana, Imagen, Midjourney, DALL·E 3, Flux, SD от 1.x до 3.5, файнтюны с Civitai и кадры видеомоделей. Поэтому реальные фото и фейки похожи по сюжетам, и отличить их по содержанию нельзя.

| Сплит | Строк | Объём | Что внутри |
|---|---|---|---|
| core/train | 2.3M | 3.3 TB | реальные из LAION и Pexels, 72 генератора |
| core/test | 91k | 68 GB | только новые генераторы: GPT Image 1.5 и 2, Nano Banana Pro, Midjourney 7, Flux.2 klein 9B, Z-Image Turbo и другие; реальные из DOCCI и ImageNet |

**Что берём.** Файлы train скачиваются целиком, из каждого берутся нужные картинки, и файл удаляется. Это все 608 файлов: до `n_per_model` картинок на каждый генератор из списка `generators` в конфиге и `n_real` реальных. Генераторы очень неравны: у SD 3.5 около 150k картинок, у Nano Banana 2 около двух тысяч, и у таких берём всё, что попалось. Модель `tiny-random-sana` выкидываем, её картинки сплошной шум. Загрузчик сразу кладёт картинки по папкам: `data/real/<источник>/` и `data/fake/<генератор>/`. Туда же попадает test OpenFake с новыми генераторами: он тоже идёт в обучение. Свои папки с реальными картинками можно положить в `data/real/` или перечислить в `real_dirs`, они читаются со всеми подпапками.

**Разбиение.** Каждая картинка по хешу имени папки и файла попадает в одну из трёх частей: 85 % обучение, 5 % валидация (лучшая эпоха и калибровка), 10 % тест (только отчёт). Разбиение одинаково при каждом запуске, и картинка остаётся в своей части, когда добавляются новые. К реальным обучения добавляются отрисованные кодом тексты, интерфейсы, графики и логотипы, 5 % от реальных (`render_train`): так модель реже срабатывает на скриншотах и документах. Цена: сгенерированные логотипы и инфографику она ловит хуже. На тесте у таких картинок другие шрифты.

**Хранение и обработка.** Картинки лежат на диске как есть, байт в байт из датасета: фейки в основном PNG, реальные фото в основном JPEG. Картинки меньше 512 пикселей пропускаются. Картинка, которая лежит не в JPEG, сначала один раз сжимается в JPEG, как будто её уже публиковали: так история сжатия есть у обоих классов. Дальше по порядку, одинаково для обоих классов, как с картинками бывает в интернете: случайный кусок на 50–100 % площади с пропорциями от 1:2 до 2:1, какими бы ни были пропорции самой картинки (так растяжение до квадрата одинаково у обоих классов); в исходном разрешении иногда цвет (яркость, контраст, насыщенность), резкость, цензура области (размытие, пикселизация, плашка или полоса на 1–20 % картинки), пересэмплинг, размытие и шум, затем сжатие в JPEG или WebP со случайным качеством от 50 до 95, иногда дважды. Треть кадров получает только кусок и сжатие. Только после этого картинка целиком сжимается до 512×512 без сохранения пропорций, как обычно у моделей зрения, половина отражается, 5 % поворачиваются на 90°. Все кадры одного размера, поэтому батчи обычные, по 32.

**Эпоха.** Все сгенерированные картинки по одному разу и столько же реальных, каждый раз новых. С `train.balance` больше нуля сгенерированные набираются с весами, и картинки редких генераторов повторяются.

**Тест.** Картинка целиком сжимается до 512×512, как на обучении, ничего не обрезается.
- **Тестовая часть, 10 %:** ложные срабатывания по каждому источнику реальных и доля пойманных по каждому генератору. Эти генераторы модель видела, новых среди них нет.
- **SID_Set:** другой датасет, на нём модель не училась: свои реальные, полностью сгенерированные картинки и локальные правки. Это проверка на незнакомых данных.
- **Отрисованные тексты, интерфейсы, графики и логотипы:** сколько ложных срабатываний на нефото.

In [ ]:
# сколько картинок от каждого генератора
from src.data import distribution, load_pool

parts = load_pool(cfg['data'])
rows = [r for part in parts.values() for r in part]
print(distribution(parts, cfg['model']['families']))

In [ ]:
# 5 случайных картинок одного генератора
import random
from PIL import Image

gen = 'gpt-image-1'  # любое имя из списка выше, или 'real'
pool = [r for r in rows if (r['generator'] or 'real') == gen]
pick = random.sample(pool, min(5, len(pool)))
fig, ax = plt.subplots(1, len(pick), figsize=(4 * len(pick), 4), squeeze=False)
for a, r in zip(ax[0], pick):
    img = Image.open(r['path']).convert('RGB')
    a.imshow(img)
    a.set_title(f'{gen}\n{img.width}×{img.height}', fontsize=9)
    a.axis('off')